In [22]:
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer, GPT2Model, GPT2LMHeadModel
from torch.nn.functional import softmax
from datasets import load_dataset


In [23]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

In [32]:
LABELS = ['negative', 'positive']
TOKENIZED_LABELS = list(tokenizer(LABELS, return_tensors='pt', padding=True, truncation=True)['input_ids'].squeeze().tolist())
LABEL_MAPPING = {'negative': 0, 'positive': 1}


[31591, 24561]


In [85]:
imdb = load_dataset('imdb')
imdb

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [91]:
imdb_test = imdb['test'].shuffle().select(range(1000))
imdb_test['label']

[0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 1,


In [ ]:
class SimpleGPT2SequenceClassifier(nn.Module):
    def __init__(self, hidden_size: int, num_classes:int ,max_seq_len:int, gpt_model_name:str):
        super(SimpleGPT2SequenceClassifier,self).__init__()
        self.gpt2model = GPT2Model.from_pretrained(gpt_model_name)
        self.fc1 = nn.Linear(hidden_size*max_seq_len, num_classes)

        
    def forward(self, input_id, mask):
        gpt_out, _ = self.gpt2model(input_ids=input_id, attention_mask=mask, return_dict=False)
        batch_size = gpt_out.shape[0]
        linear_output = self.fc1(gpt_out.view(batch_size,-1))
        return linear_output
    
head_classifier = SimpleGPT2SequenceClassifier(hidden_size=512, num_classes=2, max_seq_len=1024, gpt_model_name="gpt2")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import GPT2Tokenizer, GPT2Model
from datasets import load_dataset
from tqdm import tqdm

class SimpleGPT2SequenceClassifier(nn.Module):
    def __init__(self, hidden_size: int, num_classes: int, max_seq_len: int, gpt_model_name: str):
        super(SimpleGPT2SequenceClassifier, self).__init__()
        self.gpt2model = GPT2Model.from_pretrained(gpt_model_name)
        self.fc1 = nn.Linear(hidden_size * max_seq_len, num_classes)

    def forward(self, input_id, mask):
        gpt_out = self.gpt2model(input_ids=input_id, attention_mask=mask).last_hidden_state
        batch_size = gpt_out.shape[0]
        linear_output = self.fc1(gpt_out.view(batch_size, -1))
        return linear_output

# Load the IMDB dataset
dataset = load_dataset('imdb')

# Load the GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
max_seq_len = 1024

# Preprocess the dataset
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=max_seq_len)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Convert dataset to PyTorch tensors
class IMDBDataset(Dataset):
    def __init__(self, tokenized_dataset):
        self.tokenized_dataset = tokenized_dataset

    def __len__(self):
        return len(self.tokenized_dataset)

    def __getitem__(self, idx):
        item = self.tokenized_dataset[idx]
        input_ids = torch.tensor(item['input_ids'])
        attention_mask = torch.tensor(item['attention_mask'])
        label = torch.tensor(item['label'])
        return input_ids, attention_mask, label

train_dataset = IMDBDataset(tokenized_datasets['train'])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# Define the model, loss function, and optimizer
hidden_size = 768  # GPT-2 hidden size
num_classes = 2
model = SimpleGPT2SequenceClassifier(hidden_size=hidden_size, num_classes=num_classes, max_seq_len=max_seq_len, gpt_model_name='gpt2')
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)

# Training loop
num_epochs = 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for input_ids, attention_mask, labels in tqdm(train_loader):
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {avg_loss}")

# Evaluation loop
model.eval()
total_correct = 0
total_samples = 0

In [92]:
def classify_sentiment(model, sample_text):
    # Prepare the input text with the sentiment prompt
    input_text = sample_text + '\n\n the sentiment of this review is '
    
    # Encode the input text
    inputs = tokenizer(input_text, return_tensors='pt', max_length=1024, truncation=True)
    
    # Forward pass through the model to get logits
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get logits from the model's output
    logits = outputs.logits[:, -1, :]  # Logits for the last token
    
    # Apply softmax to get probabilities
    probabilities = softmax(logits, dim=-1)

    # get the highest probability index of all the tokens listed in TOKENIZED_LABELS

    probabilities = probabilities[0][TOKENIZED_LABELS]
    
    # Get the predicted class based on the highest probability
    predicted_idx = torch.argmax(probabilities).item()
    predicted_sentiment = LABELS[predicted_idx]
    
    return predicted_sentiment

def classify_sentiment_generate(model, sample_text):
    # Preprocess the text and append the classification prompt
    input_text = sample_text + '\n\n the sentiment of this review is '
    inputs = tokenizer(input_text, return_tensors='pt', max_length=1024, truncation=True)
    
    # Generate output from the model
    outputs = model.generate(**inputs, max_length=inputs['input_ids'].shape[1] + 1, num_return_sequences=1)
    
    # Decode the generated text
    decoded_output = tokenizer.decode(outputs[0])
    
    # Check for sentiment labels in the output
    if decoded_output.split()[-1].lower() in LABELS:
        return decoded_output.split()[-1]
    
    print(decoded_output)
    return classify_sentiment_random(model, sample_text)
    
def classify_sentiment_random(model, sample_text):
    if torch.rand(1) < 0.5:
        return 'positive'

    return 'negative'

In [93]:

def test_classification_methods(model, dataset, methods):
    results = {method_name: {'correct': 0, 'total': 0} for method_name in methods.keys()}

    for sample in dataset:
        text = sample['text']
        true_label = 'positive' if sample['label'] == 1 else 'negative'

        for method_name, method in methods.items():
            predicted_label = method(model, text)
            results[method_name]['correct'] += 1 if predicted_label == true_label else 0
            results[method_name]['total'] += 1

    for method_name in results.keys():
        accuracy = results[method_name]['correct'] / results[method_name]['total']
        print(f"{method_name} accuracy: {accuracy:.2%}")

# Define the methods dictionary
methods = {
    'logits': classify_sentiment,
    # 'generate': classify_sentiment_generate,
    'random': classify_sentiment_random
}

# Test the classification methods on the first 100 samples
test_classification_methods(model, imdb_test, methods)

logits accuracy: 65.00%
random accuracy: 49.90%
